In [1]:
import os
import scanpy as sc
import pandas as pd
import scanpy.external as sce

# Preprocess

In [2]:
folder_path = '../3_rna_atac_cell/data_rna/'
mtx_files = [os.path.join(folder_path, f) for f in os.listdir(folder_path) if f.endswith('.mtx')]

In [3]:
adata_list = [sc.read_mtx(mtx_file) for mtx_file in mtx_files]

In [4]:
folder_path = '../3_rna_atac_cell/data_rna/'
csv_files = [os.path.join(folder_path, f) for f in os.listdir(folder_path) if f.startswith('cell')]

In [5]:
cell_list = [pd.read_csv(csv_file, index_col=0) for csv_file in csv_files]

In [6]:
adata_list

[AnnData object with n_obs × n_vars = 36601 × 1228240,
 AnnData object with n_obs × n_vars = 36601 × 453606,
 AnnData object with n_obs × n_vars = 36601 × 874480]

In [8]:
for i in range(len(adata_list)):
    adata_list[i].var_names = cell_list[i]['cell_id'].tolist()
    adata_list[i] = adata_list[i].T

In [29]:
combined = adata_list[0].concatenate(adata_list[1], index_unique=None)

/tmp/ipykernel_2594694/1589350125.py:1: FutureWarning: Use anndata.concat instead of AnnData.concatenate, AnnData.concatenate is deprecated and will be removed in the future. See the tutorial for concat at: https://anndata.readthedocs.io/en/latest/concatenation.html
  combined = adata_list[0].concatenate(adata_list[1], index_unique=None)


In [30]:
combined = combined.concatenate(adata_list[2], index_unique=None)

/tmp/ipykernel_2594694/565997766.py:1: FutureWarning: Use anndata.concat instead of AnnData.concatenate, AnnData.concatenate is deprecated and will be removed in the future. See the tutorial for concat at: https://anndata.readthedocs.io/en/latest/concatenation.html
  combined = combined.concatenate(adata_list[2], index_unique=None)


In [33]:
gene = pd.read_csv('../3_rna_atac_cell/data_rna/gene.csv', index_col=0)
gene

,gene
1,MIR1302-2HG
2,FAM138A
3,OR4F5
4,AL627309.1
5,AL627309.3
...,...
36597,AC141272.1
36598,AC023491.2
36599,AC007325.1
36600,AC007325.4


In [34]:
combined.var_names = gene['gene'].tolist()

In [35]:
del combined.obs['batch']
combined

AnnData object with n_obs × n_vars = 2556326 × 36601

In [41]:
df = pd.DataFrame([x.split('-') for x in combined.obs_names], columns=['part1', 'part2', 'part3'])

In [42]:
df

,part1,part2,part3
0,10x_datasets_1,AAACAGCCAACAACAA,1
1,10x_datasets_1,AAACAGCCACCGGCTA,1
2,10x_datasets_1,AAACAGCCAGGACACA,1
3,10x_datasets_1,AAACAGCCATCCTAGA,1
4,10x_datasets_1,AAACATGCAAAGGTAC,1
...,...,...,...
2556321,Hu_BioRxiv_2023,T124.R1.192.R2.191.R3.121.P1.04,donor7
2556322,Hu_BioRxiv_2023,T124.R1.192.R2.191.R3.135.P1.04,donor7
2556323,Hu_BioRxiv_2023,T124.R1.192.R2.192.R3.001.P1.04,donor7
2556324,Hu_BioRxiv_2023,T124.R1.192.R2.192.R3.033.P1.04,donor7


In [71]:
combined.obs['dataset'] = df['part1'].tolist()
combined.obs['barcode'] = df['part2'].tolist()

In [77]:
df['part4'] = df['part1'] + '-' + df['part3']

In [78]:
df

,part1,part2,part3,part4
0,10x_datasets_1,AAACAGCCAACAACAA,1,10x_datasets_1-1
1,10x_datasets_1,AAACAGCCACCGGCTA,1,10x_datasets_1-1
2,10x_datasets_1,AAACAGCCAGGACACA,1,10x_datasets_1-1
3,10x_datasets_1,AAACAGCCATCCTAGA,1,10x_datasets_1-1
4,10x_datasets_1,AAACATGCAAAGGTAC,1,10x_datasets_1-1
...,...,...,...,...
2556321,Hu_BioRxiv_2023,T124.R1.192.R2.191.R3.121.P1.04,donor7,Hu_BioRxiv_2023-donor7
2556322,Hu_BioRxiv_2023,T124.R1.192.R2.191.R3.135.P1.04,donor7,Hu_BioRxiv_2023-donor7
2556323,Hu_BioRxiv_2023,T124.R1.192.R2.192.R3.001.P1.04,donor7,Hu_BioRxiv_2023-donor7
2556324,Hu_BioRxiv_2023,T124.R1.192.R2.192.R3.033.P1.04,donor7,Hu_BioRxiv_2023-donor7


In [79]:
combined.obs['sample'] = df['part4'].tolist()

In [84]:
combined.write('rna.h5ad', compression='gzip')

In [3]:
adata = sc.read_h5ad('./rna.h5ad')
adata

AnnData object with n_obs × n_vars = 2556326 × 36601
    obs: 'dataset', 'barcode', 'sample'

In [5]:
ann = pd.read_csv('../4_cell_ann/ann_merge.csv', index_col=0)
ann

,cellType
Chen_Nat_Methods_2022-AAACAGCCAACTGGCT-lane1,Treg
Chen_Nat_Methods_2022-AAACATGCACAATTAC-lane1,Treg
Chen_Nat_Methods_2022-AAACATGCAGCACGTT-lane1,Th17
Chen_Nat_Methods_2022-AAACATGCATATTGAC-lane1,Th1
Chen_Nat_Methods_2022-AAACCAACAATAGCAA-lane1,TCM
...,...
Wimmers_Cell_2023-TTTGTCTAGTTGTCCC-76_CC0829,CD14 Monocyte
Wimmers_Cell_2023-TTTGTGGCAGAGAGCC-76_CC0829,CD8 T Naive
Wimmers_Cell_2023-TTTGTGGCATTGCAGC-76_CC0829,PB
Wimmers_Cell_2023-TTTGTGGCATTGCGAC-76_CC0829,CD14 Monocyte


In [23]:
df = pd.DataFrame([x.split('-') for x in ann.index], columns=['part1', 'part2', 'part3'])
df['part4'] = df['part1'] + '-' + df['part3']

In [24]:
ann['dataset'] = df['part1'].tolist()
ann['sample'] = df['part4'].tolist()

In [25]:
ann

,cellType,dataset,sample
Chen_Nat_Methods_2022-AAACAGCCAACTGGCT-lane1,Treg,Chen_Nat_Methods_2022,Chen_Nat_Methods_2022-lane1
Chen_Nat_Methods_2022-AAACATGCACAATTAC-lane1,Treg,Chen_Nat_Methods_2022,Chen_Nat_Methods_2022-lane1
Chen_Nat_Methods_2022-AAACATGCAGCACGTT-lane1,Th17,Chen_Nat_Methods_2022,Chen_Nat_Methods_2022-lane1
Chen_Nat_Methods_2022-AAACATGCATATTGAC-lane1,Th1,Chen_Nat_Methods_2022,Chen_Nat_Methods_2022-lane1
Chen_Nat_Methods_2022-AAACCAACAATAGCAA-lane1,TCM,Chen_Nat_Methods_2022,Chen_Nat_Methods_2022-lane1
...,...,...,...
Wimmers_Cell_2023-TTTGTCTAGTTGTCCC-76_CC0829,CD14 Monocyte,Wimmers_Cell_2023,Wimmers_Cell_2023-76_CC0829
Wimmers_Cell_2023-TTTGTGGCAGAGAGCC-76_CC0829,CD8 T Naive,Wimmers_Cell_2023,Wimmers_Cell_2023-76_CC0829
Wimmers_Cell_2023-TTTGTGGCATTGCAGC-76_CC0829,PB,Wimmers_Cell_2023,Wimmers_Cell_2023-76_CC0829
Wimmers_Cell_2023-TTTGTGGCATTGCGAC-76_CC0829,CD14 Monocyte,Wimmers_Cell_2023,Wimmers_Cell_2023-76_CC0829


In [26]:
ann_filter = ann.dropna(subset=['cellType']) # 删去NA值
ann_filter

,cellType,dataset,sample
Chen_Nat_Methods_2022-AAACAGCCAACTGGCT-lane1,Treg,Chen_Nat_Methods_2022,Chen_Nat_Methods_2022-lane1
Chen_Nat_Methods_2022-AAACATGCACAATTAC-lane1,Treg,Chen_Nat_Methods_2022,Chen_Nat_Methods_2022-lane1
Chen_Nat_Methods_2022-AAACATGCAGCACGTT-lane1,Th17,Chen_Nat_Methods_2022,Chen_Nat_Methods_2022-lane1
Chen_Nat_Methods_2022-AAACATGCATATTGAC-lane1,Th1,Chen_Nat_Methods_2022,Chen_Nat_Methods_2022-lane1
Chen_Nat_Methods_2022-AAACCAACAATAGCAA-lane1,TCM,Chen_Nat_Methods_2022,Chen_Nat_Methods_2022-lane1
...,...,...,...
Wimmers_Cell_2023-TTTGTCTAGTTGTCCC-76_CC0829,CD14 Monocyte,Wimmers_Cell_2023,Wimmers_Cell_2023-76_CC0829
Wimmers_Cell_2023-TTTGTGGCAGAGAGCC-76_CC0829,CD8 T Naive,Wimmers_Cell_2023,Wimmers_Cell_2023-76_CC0829
Wimmers_Cell_2023-TTTGTGGCATTGCAGC-76_CC0829,PB,Wimmers_Cell_2023,Wimmers_Cell_2023-76_CC0829
Wimmers_Cell_2023-TTTGTGGCATTGCGAC-76_CC0829,CD14 Monocyte,Wimmers_Cell_2023,Wimmers_Cell_2023-76_CC0829


In [42]:
test = ann_filter[ann_filter['cellType']=='Proerythroblast']
test

,cellType,dataset,sample
NeurIPS_2021-GGTTGCGGTTGCTTCG-site1_donor1,Proerythroblast,NeurIPS_2021,NeurIPS_2021-site1_donor1
NeurIPS_2021-CCTTCAATCACCTGCT-site1_donor1,Proerythroblast,NeurIPS_2021,NeurIPS_2021-site1_donor1
NeurIPS_2021-GGTCCTGCATCTTGAG-site1_donor1,Proerythroblast,NeurIPS_2021,NeurIPS_2021-site1_donor1
NeurIPS_2021-CCTTAACGTGCTAGCG-site1_donor1,Proerythroblast,NeurIPS_2021,NeurIPS_2021-site1_donor1
NeurIPS_2021-TATGACATCGTTACTT-site1_donor1,Proerythroblast,NeurIPS_2021,NeurIPS_2021-site1_donor1
...,...,...,...
NeurIPS_2021-TCCAGGATCACCATTT-site4_donor9,Proerythroblast,NeurIPS_2021,NeurIPS_2021-site4_donor9
NeurIPS_2021-TGTAAGCTCGCTAAGT-site4_donor9,Proerythroblast,NeurIPS_2021,NeurIPS_2021-site4_donor9
NeurIPS_2021-CTCACACTCGGGACCT-site4_donor9,Proerythroblast,NeurIPS_2021,NeurIPS_2021-site4_donor9
NeurIPS_2021-CCAGGATGTATTCGTC-site4_donor9,Proerythroblast,NeurIPS_2021,NeurIPS_2021-site4_donor9


In [41]:
test['cellType'].value_counts()

cellType
CD8+ T                 11589
CD14+ Mono             10843
NK                      6929
CD4+ T activated        5526
Naive CD20+ B           5052
Erythroblast            4916
CD4+ T naive            4398
Transitional B          2810
Proerythroblast         2300
CD16+ Mono              1894
B1 B                    1890
Normoblast              1780
Lymph prog              1779
G/M prog                1203
pDC                     1191
HSC                     1072
CD8+ T naive            1012
MK/E prog                884
cDC2                     859
ILC                      835
Plasma cell              379
ID2-hi myeloid prog      108
Name: count, dtype: int64

In [45]:
ann_all = pd.DataFrame(adata.obs['sample'].value_counts())
ann_all

,count
sample,
Hu_BioRxiv_2023-donor6,188795
Hu_BioRxiv_2023-donor1,163306
Hu_BioRxiv_2023-donor2,126318
Hu_BioRxiv_2023-donor3,115336
Hu_BioRxiv_2023-donor4,110777
...,...
Wimmers_Cell_2023-76_CC0829,2023
NeurIPS_2021-site3_donor7,1771
NeurIPS_2021-site3_donor6,1679


In [46]:
ann_all.to_csv('./processed/original_ann/sample_all.csv')

# Try harmony

In [ ]:
adata = sc.read_h5ad('./processed/all_cell/rna_leiden.h5ad')
adata

In [4]:
# sc.pp.calculate_qc_metrics(adata,inplace=True)
# sc.pp.normalize_total(adata, target_sum=1e4)
# sc.pp.log1p(adata)
# sc.pp.highly_variable_genes(adata, min_mean=0.0125, max_mean=3, min_disp=0.5)
# adata.raw = adata
# adata = adata[:, adata.var.highly_variable]
# sc.pp.scale(adata, max_value=10)
# sc.tl.pca(adata, svd_solver='arpack')
# sc.pl.pca_variance_ratio(adata, log=True)

# sce.pp.harmony_integrate(adata, 'sample')
# adata.obsm['X_pca'] = adata.obsm['X_pca_harmony']
# sc.pp.neighbors(adata, n_neighbors=10, n_pcs=40)

# adata.write('rna_neighbor.h5ad', compression='gzip')

# sc.tl.umap(adata)
# sc.tl.leiden(adata,resolution=1)

# adata.write('rna_leiden.h5ad', compression='gzip')

In [2]:
# sc.pl.umap(adata, color='dataset')

In [12]:
sc.tl.leiden(adata,resolution=0.05)

IOStream.flush timed out
IOStream.flush timed out


In [1]:
# sc.pl.umap(adata, color='leiden', save='umap')

In [3]:
# sc.pl.dotplot(adata, 
#               groupby='leiden', 
#               var_names=[
#                          # 'LEPR','CXCL12','NCAM1', # Mesenchymal
#                          # 'TAGLN','ACTA2', # Vascular smooth muscle
#                          # 'EMCN', 'FLT1', # Endothelial cell
#                          'S100A9','LYZ','SRGN',# Myeloid
#                          'MZB1','GZMA','IL7R','MPO','DNTT','CD79B','PAX5', # Lymphoid
#                          'ITGA2B','HBA1','HBA2','GATA1', # Megakaryocyte/Erythroid
#                          'AVP','SPINK2' # HSPC
#                         ])

In [15]:
adata.obs['leiden'].value_counts()

leiden
0    1626981
1     402920
2     201490
3     185192
4     139461
5        138
6         95
7         49
Name: count, dtype: int64

In [25]:
adata_1 = adata[adata.obs['leiden'] == '1']

In [26]:
adata_1

View of AnnData object with n_obs × n_vars = 402920 × 6066
    obs: 'dataset', 'barcode', 'sample', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'pct_counts_in_top_50_genes', 'pct_counts_in_top_100_genes', 'pct_counts_in_top_200_genes', 'pct_counts_in_top_500_genes', 'leiden'
    var: 'n_cells_by_counts', 'mean_counts', 'log1p_mean_counts', 'pct_dropout_by_counts', 'total_counts', 'log1p_total_counts', 'highly_variable', 'means', 'dispersions', 'dispersions_norm', 'mean', 'std'
    uns: 'hvg', 'leiden', 'log1p', 'neighbors', 'pca', 'umap', 'dataset_colors', 'leiden_colors'
    obsm: 'X_pca', 'X_pca_harmony', 'X_umap'
    varm: 'PCs'
    obsp: 'connectivities', 'distances'

In [29]:
adata.obs['dataset'].value_counts()

dataset
Hu_BioRxiv_2023                874480
Thomson_Nat_Immunol_2024       336485
Wimmers_Cell_2023              316906
Cheong_Cell_2023_1             197360
Oguchi_Science_2024            145485
Wei_Immuni_2023                143804
Boukhaled_Nat_Immunol_2022     101020
NeurIPS_2021                    69249
Mimitou_Nat_Biotechnol_2021     39374
Weng_Nature_2024_5              29414
Lareau_Nat_Genet_2023           29363
Swanson_Elife_2021_1            25517
Weng_Nature_2024_4              25442
Weng_Nature_2024_6              23154
Li_Nat_Biotechnol_2023          19260
Weng_Nature_2024_3              18585
Weng_Nature_2024_2              17310
Cheong_Cell_2023_2              17244
Weng_Nature_2024_1              14917
Swanson_Elife_2021_2            14680
10x_datasets_8                  12016
10x_datasets_6                  12012
10x_datasets_10                 11909
10x_datasets_4                  11898
10x_datasets_1                  10970
10x_datasets_2                  10691
Chen

In [28]:
adata_1.obs['dataset'].value_counts()

dataset
Hu_BioRxiv_2023                402183
NeurIPS_2021                      115
Li_Nat_Biotechnol_2023            111
Boukhaled_Nat_Immunol_2022         92
Weng_Nature_2024_5                 83
Weng_Nature_2024_6                 63
Weng_Nature_2024_4                 51
Weng_Nature_2024_1                 47
Cheong_Cell_2023_1                 42
Weng_Nature_2024_7                 33
Weng_Nature_2024_2                 25
Weng_Nature_2024_3                 20
Wei_Immuni_2023                    15
10x_datasets_8                     10
Mimitou_Nat_Biotechnol_2021         9
Wimmers_Cell_2023                   7
Cheong_Cell_2023_2                  6
10x_datasets_6                      4
10x_datasets_2                      1
Lareau_Nat_Genet_2023               1
10x_datasets_7                      1
10x_datasets_1                      1
Name: count, dtype: int64

# Filter annotated cells using ann_filter

## Preprocess for ann_merge

In [2]:
ann = pd.read_csv('../4_cell_ann/ann_merge.csv', index_col=0)
ann

,cellType
Chen_Nat_Methods_2022-AAACAGCCAACTGGCT-lane1,Treg
Chen_Nat_Methods_2022-AAACATGCACAATTAC-lane1,Treg
Chen_Nat_Methods_2022-AAACATGCAGCACGTT-lane1,Th17
Chen_Nat_Methods_2022-AAACATGCATATTGAC-lane1,Th1
Chen_Nat_Methods_2022-AAACCAACAATAGCAA-lane1,TCM
...,...
Wimmers_Cell_2023-TTTGTCTAGTTGTCCC-76_CC0829,CD14 Monocyte
Wimmers_Cell_2023-TTTGTGGCAGAGAGCC-76_CC0829,CD8 T Naive
Wimmers_Cell_2023-TTTGTGGCATTGCAGC-76_CC0829,PB
Wimmers_Cell_2023-TTTGTGGCATTGCGAC-76_CC0829,CD14 Monocyte


In [3]:
exclude_values = ['Doublet','Doublet.B','Doublet.PB','Doublet1','Doublet2','Doublet3','Doublet4','Doublet5','lowQual']

In [4]:
ann_filter = ann[~ann['cellType'].isin(exclude_values)]
ann_filter

,cellType
Chen_Nat_Methods_2022-AAACAGCCAACTGGCT-lane1,Treg
Chen_Nat_Methods_2022-AAACATGCACAATTAC-lane1,Treg
Chen_Nat_Methods_2022-AAACATGCAGCACGTT-lane1,Th17
Chen_Nat_Methods_2022-AAACATGCATATTGAC-lane1,Th1
Chen_Nat_Methods_2022-AAACCAACAATAGCAA-lane1,TCM
...,...
Wimmers_Cell_2023-TTTGTCTAGTTGTCCC-76_CC0829,CD14 Monocyte
Wimmers_Cell_2023-TTTGTGGCAGAGAGCC-76_CC0829,CD8 T Naive
Wimmers_Cell_2023-TTTGTGGCATTGCAGC-76_CC0829,PB
Wimmers_Cell_2023-TTTGTGGCATTGCGAC-76_CC0829,CD14 Monocyte


In [5]:
ann_filter = ann_filter.dropna(subset=['cellType']) # 删去NA值
ann_filter

,cellType
Chen_Nat_Methods_2022-AAACAGCCAACTGGCT-lane1,Treg
Chen_Nat_Methods_2022-AAACATGCACAATTAC-lane1,Treg
Chen_Nat_Methods_2022-AAACATGCAGCACGTT-lane1,Th17
Chen_Nat_Methods_2022-AAACATGCATATTGAC-lane1,Th1
Chen_Nat_Methods_2022-AAACCAACAATAGCAA-lane1,TCM
...,...
Wimmers_Cell_2023-TTTGTCTAGTTGTCCC-76_CC0829,CD14 Monocyte
Wimmers_Cell_2023-TTTGTGGCAGAGAGCC-76_CC0829,CD8 T Naive
Wimmers_Cell_2023-TTTGTGGCATTGCAGC-76_CC0829,PB
Wimmers_Cell_2023-TTTGTGGCATTGCGAC-76_CC0829,CD14 Monocyte


In [6]:
ann_filter.value_counts()

cellType      
CD4 Memory        226556
NK                170535
CD4               153153
CD8               134059
CD14mono          129800
                   ...  
15_CLP.2              18
14_Unk                11
04_Early.Baso          9
11_CD14.Mono.1         8
26_Unk                 1
Name: count, Length: 143, dtype: int64

In [7]:
ann_filter.to_csv('./processed/original_ann/ann_filter.csv')

## Filter cell

In [2]:
adata = sc.read_h5ad('./processed/all_cell/rna.h5ad')
adata

KeyboardInterrupt: 

In [3]:
ann = pd.read_csv('./processed/original_ann/ann_filter.csv', index_col=0)
ann

,cellType
Chen_Nat_Methods_2022-AAACAGCCAACTGGCT-lane1,Treg
Chen_Nat_Methods_2022-AAACATGCACAATTAC-lane1,Treg
Chen_Nat_Methods_2022-AAACATGCAGCACGTT-lane1,Th17
Chen_Nat_Methods_2022-AAACATGCATATTGAC-lane1,Th1
Chen_Nat_Methods_2022-AAACCAACAATAGCAA-lane1,TCM
...,...
Wimmers_Cell_2023-TTTGTCTAGTTGTCCC-76_CC0829,CD14 Monocyte
Wimmers_Cell_2023-TTTGTGGCAGAGAGCC-76_CC0829,CD8 T Naive
Wimmers_Cell_2023-TTTGTGGCATTGCAGC-76_CC0829,PB
Wimmers_Cell_2023-TTTGTGGCATTGCGAC-76_CC0829,CD14 Monocyte


In [4]:
adata_filtered = adata[adata.obs_names.isin(ann.index)]
adata_filtered

View of AnnData object with n_obs × n_vars = 2113504 × 36601
    obs: 'dataset', 'barcode', 'sample'

In [5]:
ann_sorted = ann.reindex(adata_filtered.obs_names.tolist())
ann_sorted

,cellType
Chen_Nat_Methods_2022-AAACAGCCAACTGGCT-lane1,Treg
Chen_Nat_Methods_2022-AAACATGCACAATTAC-lane1,Treg
Chen_Nat_Methods_2022-AAACATGCAGCACGTT-lane1,Th17
Chen_Nat_Methods_2022-AAACATGCATATTGAC-lane1,Th1
Chen_Nat_Methods_2022-AAACCAACAATAGCAA-lane1,TCM
...,...
Hu_BioRxiv_2023-T124.R1.192.R2.191.R3.121.P1.04-donor7,CD4
Hu_BioRxiv_2023-T124.R1.192.R2.191.R3.135.P1.04-donor7,NK
Hu_BioRxiv_2023-T124.R1.192.R2.192.R3.001.P1.04-donor7,CD4
Hu_BioRxiv_2023-T124.R1.192.R2.192.R3.033.P1.04-donor7,CD8


In [6]:
adata_filtered.obs['cellType'] = ann_sorted['cellType'].tolist()

/tmp/ipykernel_1553911/799291639.py:1: ImplicitModificationWarning: Trying to modify attribute `.obs` of view, initializing view as actual.
  adata_filtered.obs['cellType'] = ann_sorted['cellType'].tolist()


In [8]:
adata_filtered.write_h5ad('./processed/ann_cell/ann_cell_rna.h5ad', compression='gzip')

In [18]:
rna = sc.read_h5ad('./processed/ann_cell/ann_cell_atac.h5ad')
rna

AnnData object with n_obs × n_vars = 2113504 × 527459
    obs: 'dataset', 'barcode', 'sample', 'cellType'

In [19]:
sc.pp.calculate_qc_metrics(rna,inplace=True)

In [6]:
# rna.var["mt"] = rna.var_names.str.startswith("MT-")
# sc.pp.calculate_qc_metrics(
#     rna, qc_vars=["mt"], percent_top=None, log1p=False, inplace=True
# )

In [20]:
rna

AnnData object with n_obs × n_vars = 2113504 × 527459
    obs: 'dataset', 'barcode', 'sample', 'cellType', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'pct_counts_in_top_50_genes', 'pct_counts_in_top_100_genes', 'pct_counts_in_top_200_genes', 'pct_counts_in_top_500_genes'
    var: 'n_cells_by_counts', 'mean_counts', 'log1p_mean_counts', 'pct_dropout_by_counts', 'total_counts', 'log1p_total_counts'

In [ ]:
sc.pl.violin(
    rna,
    ["total_counts"],
    size=0,
    multi_panel=True,
)

In [22]:
a = pd.DataFrame(rna.obs['total_counts'].value_counts())

In [23]:
a.to_csv('./a.csv')

## cellhint

In [4]:
adata = sc.read_h5ad('./processed/ann_cell/ann_cell_rna.h5ad')
adata

AnnData object with n_obs × n_vars = 2113504 × 36601
    obs: 'dataset', 'barcode', 'sample', 'cellType'

In [5]:
adata.obs.dataset.value_counts()

dataset
Hu_BioRxiv_2023             874480
Thomson_Nat_Immunol_2024    336485
Wimmers_Cell_2023           288211
Cheong_Cell_2023_1          197341
Oguchi_Science_2024         145485
NeurIPS_2021                 69249
Lareau_Nat_Genet_2023        29363
Weng_Nature_2024_5           28948
Swanson_Elife_2021_1         25517
Weng_Nature_2024_4           25252
Li_Nat_Biotechnol_2023       19260
Weng_Nature_2024_3           18420
Cheong_Cell_2023_2           17244
Weng_Nature_2024_2           17011
Weng_Nature_2024_1           12781
Chen_Nat_Methods_2022         8457
Name: count, dtype: int64

In [6]:
sc.pp.normalize_total(adata, target_sum = 1e4)
sc.pp.log1p(adata)
sc.pp.highly_variable_genes(adata, batch_key = 'dataset', subset = True)
sc.pp.scale(adata, max_value = 10)
sc.tl.pca(adata)
sc.pp.neighbors(adata)
sc.tl.umap(adata)

/home/qiaolibo/miniconda3/envs/scanpy/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
IOStream.flush timed out
IOStream.flush timed out
IOStream.flush timed out
IOStream.flush timed out
IOStream.flush timed out
IOStream.flush timed out
IOStream.flush timed out
IOStream.flush timed out


In [15]:
adata.write_h5ad('./processed/ann_cell/ann_cell_rna_raw_umap.h5ad', compression='gzip')

In [ ]:
sc.pl.umap(adata, color = 'dataset')

In [ ]:
sc.pl.umap(adata, color = 'cellType')

In [11]:
import cellhint

In [12]:
alignment = cellhint.harmonize(adata, 'dataset', 'cellType')

👀 Detected PCA coordinates in the object, will use these to calculate distances
🏆 Reordering datasets
🖇 Harmonizing cell types of Weng_Nature_2024_1 and Weng_Nature_2024_2
🖇 Harmonizing cell types of Weng_Nature_2024_5
🖇 Harmonizing cell types of Weng_Nature_2024_3
🖇 Harmonizing cell types of Weng_Nature_2024_4
🖇 Harmonizing cell types of NeurIPS_2021
🖇 Harmonizing cell types of Cheong_Cell_2023_2
🖇 Harmonizing cell types of Wimmers_Cell_2023
🖇 Harmonizing cell types of Cheong_Cell_2023_1
🖇 Harmonizing cell types of Lareau_Nat_Genet_2023
🖇 Harmonizing cell types of Swanson_Elife_2021_1
🖇 Harmonizing cell types of Oguchi_Science_2024
🖇 Harmonizing cell types of Hu_BioRxiv_2023
🖇 Harmonizing cell types of Chen_Nat_Methods_2022
🖇 Harmonizing cell types of Thomson_Nat_Immunol_2024
🖇 Harmonizing cell types of Li_Nat_Biotechnol_2023
🖋️ Reannotating cells
✅ Harmonization done!


In [13]:
alignment

Cross-dataset cell type alignment for 16 datasets
    base_distance: a cross-dataset distance object
    aligned_datasets: 'Weng_Nature_2024_1', 'Weng_Nature_2024_2', 'Weng_Nature_2024_5', 'Weng_Nature_2024_3', 'Weng_Nature_2024_4', 'NeurIPS_2021', 'Cheong_Cell_2023_2', 'Wimmers_Cell_2023', 'Cheong_Cell_2023_1', 'Lareau_Nat_Genet_2023', 'Swanson_Elife_2021_1', 'Oguchi_Science_2024', 'Hu_BioRxiv_2023', 'Chen_Nat_Methods_2022', 'Thomson_Nat_Immunol_2024', 'Li_Nat_Biotechnol_2023'
    relation: data frame of the harmonization table
    reannotation: data frame of the reannotated cells ('dataset', 'cell_type', 'reannotation', 'group')

In [14]:
alignment.write('./processed/ann_cell/alignment.pkl')

In [40]:
alignment = cellhint.DistanceAlignment.load('./processed/ann_cell/alignment.pkl')

In [16]:
alignment.relation.head(10)

,Weng_Nature_2024_1,relation,Weng_Nature_2024_2,relation,Weng_Nature_2024_5,relation,Weng_Nature_2024_3,relation,Weng_Nature_2024_4,relation,...,relation,Oguchi_Science_2024,relation,Hu_BioRxiv_2023,relation,Chen_Nat_Methods_2022,relation,Thomson_Nat_Immunol_2024,relation,Li_Nat_Biotechnol_2023
0,B,=,B,=,B,=,B,=,B,∋,...,=,NONE,=,NONE,=,NONE,=,NONE,=,NONE
1,B,=,B,=,B,=,B,=,B,∋,...,=,NONE,=,NONE,=,NONE,=,UNRESOLVED,=,NONE
2,CD4,=,CD4,=,CD4,=,CD4,=,CD4,∋,...,∋,Th17,=,NONE,=,NONE,=,NONE,=,NONE
3,CD4,=,CD4,=,CD4,=,CD4,=,CD4,∋,...,∋,Th2_Th17,=,NONE,=,NONE,=,NONE,=,NONE
4,CD4,=,CD4,=,CD4,=,CD4,=,CD4,∋,...,∋,Tfh_3,=,NONE,=,NONE,=,UNRESOLVED,=,NONE
5,CD4,=,CD4,=,CD4,=,CD4,=,CD4,∋,...,∋,Treg_2,=,NONE,=,NONE,=,UNRESOLVED,=,NONE
6,CD4,=,CD4,=,CD4,=,CD4,=,CD4,∋,...,=,UNRESOLVED,=,NONE,=,NONE,=,NONE,=,NONE
7,CD4,=,CD4,=,CD4,=,CD4,=,CD4,∋,...,=,UNRESOLVED,=,NONE,=,NONE,=,NONE,=,NONE
8,CD4,=,CD4,=,CD4,=,CD4,=,CD4,∋,...,=,NONE,=,NONE,=,NONE,=,NONE,=,NONE
9,CD8,∋,CD8,=,CD8,=,CD8,=,CD8,=,...,=,NONE,=,NONE,=,NONE,=,NONE,=,NONE


In [17]:
alignment.groups

array(['Group1', 'Group1', 'Group2', 'Group2', 'Group2', 'Group2',
       'Group2', 'Group2', 'Group2', 'Group3', 'Group3', 'Group3',
       'Group4', 'Group4', 'Group4', 'Group5', 'Group6', 'Group6',
       'Group6', 'Group4', 'Group4', 'Group7', 'Group5', 'Group5',
       'Group4', 'Group4', 'Group7', 'Group7', 'Group1', 'Group8',
       'Group9', 'Group10', 'Group11', 'Group11', 'Group12', 'Group13',
       'Group14', 'Group15', 'Group16', 'Group17', 'Group4', 'Group4',
       'Group4', 'Group4', 'Group4', 'Group7', 'Group18', 'Group19',
       'Group20', 'Group20', 'Group21', 'Group2', 'Group22', 'Group23',
       'Group22', 'Group22', 'Group22', 'Group22', 'Group23', 'Group24',
       'Group24', 'Group22', 'Group25', 'Group26', 'Group3', 'Group27',
       'Group27', 'Group28', 'Group29', 'Group30', 'Group31', 'Group29',
       'Group32', 'Group33', 'Group34', 'Group35', 'Group36', 'Group10',
       'Group37', 'Group38', 'Group39', 'Group39', 'Group39', 'Group40',
       'Group41',

In [18]:
alignment.reannotation

,dataset,cell_type,reannotation,group
ID,,,,
Chen_Nat_Methods_2022-AAACAGCCAACTGGCT-lane1,Chen_Nat_Methods_2022,Treg,UNRESOLVED = UNRESOLVED = UNRESOLVED = UNRESOL...,Group39
Chen_Nat_Methods_2022-AAACATGCACAATTAC-lane1,Chen_Nat_Methods_2022,Treg,UNRESOLVED = UNRESOLVED = UNRESOLVED = UNRESOL...,Group39
Chen_Nat_Methods_2022-AAACATGCAGCACGTT-lane1,Chen_Nat_Methods_2022,Th17,UNRESOLVED = UNRESOLVED = UNRESOLVED = UNRESOL...,Group43
Chen_Nat_Methods_2022-AAACATGCATATTGAC-lane1,Chen_Nat_Methods_2022,Th1,NONE = NONE = NONE = NONE = NONE = NONE = NONE...,Group11
Chen_Nat_Methods_2022-AAACCAACAATAGCAA-lane1,Chen_Nat_Methods_2022,TCM,NONE = NONE = NONE = NONE = NONE = NONE = NONE...,Group10
...,...,...,...,...
Hu_BioRxiv_2023-T124.R1.192.R2.191.R3.121.P1.04-donor7,Hu_BioRxiv_2023,CD4,UNRESOLVED = UNRESOLVED = UNRESOLVED = UNRESOL...,Group34
Hu_BioRxiv_2023-T124.R1.192.R2.191.R3.135.P1.04-donor7,Hu_BioRxiv_2023,NK,UNRESOLVED = UNRESOLVED = UNRESOLVED = UNRESOL...,Group43
Hu_BioRxiv_2023-T124.R1.192.R2.192.R3.001.P1.04-donor7,Hu_BioRxiv_2023,CD4,UNRESOLVED = UNRESOLVED = UNRESOLVED = UNRESOL...,Group34


In [ ]:
cellhint.treeplot(alignment, save='cellhint.pdf')

In [ ]:
cellhint.treeplot(alignment, order_dataset = True, save='cellhint_order.pdf')

In [24]:
dist_mat = alignment.base_distance.to_meta()
dist_mat.iloc[:5, :5]

,Chen_Nat_Methods_2022: Act,Chen_Nat_Methods_2022: TCM,Chen_Nat_Methods_2022: Th1,Chen_Nat_Methods_2022: Th17,Chen_Nat_Methods_2022: Th2
Chen_Nat_Methods_2022: Act,0.103352,0.110738,0.134054,0.116150,0.123198
Chen_Nat_Methods_2022: TCM,0.110738,0.091255,0.109405,0.103649,0.099801
Chen_Nat_Methods_2022: Th1,0.134054,0.109405,0.103159,0.113534,0.107180
Chen_Nat_Methods_2022: Th17,0.116150,0.103649,0.113534,0.099722,0.106126
Chen_Nat_Methods_2022: Th2,0.123198,0.099801,0.107180,0.106126,0.093569


In [25]:
member_mat = alignment.base_distance.to_meta(turn_binary = True)
member_mat.iloc[:5, :5]

,Chen_Nat_Methods_2022: Act,Chen_Nat_Methods_2022: TCM,Chen_Nat_Methods_2022: Th1,Chen_Nat_Methods_2022: Th17,Chen_Nat_Methods_2022: Th2
Chen_Nat_Methods_2022: Act,0.525862,0.104530,0.112350,0.141502,0.067071
Chen_Nat_Methods_2022: TCM,0.104530,0.240239,0.155925,0.082425,0.125893
Chen_Nat_Methods_2022: Th1,0.112350,0.155925,0.431498,0.151762,0.152922
Chen_Nat_Methods_2022: Th17,0.141502,0.082425,0.151762,0.289886,0.105125
Chen_Nat_Methods_2022: Th2,0.067071,0.125893,0.152922,0.105125,0.305882


In [ ]:
import seaborn as sns
g = sns.clustermap(member_mat)
g.savefig("cellhint_clustermap.pdf", format="pdf")

In [ ]:
cellhint.treeplot(alignment.relation[alignment.groups == 'Group1'])

In [49]:
cellhint.integrate(adata, 'sample', 'low_hierarchy')

👀 `use_rep` is not specified, will use `'X_pca'` as the search space


In [50]:
sc.tl.umap(adata)

IOStream.flush timed out
IOStream.flush timed out
IOStream.flush timed out
IOStream.flush timed out
IOStream.flush timed out
IOStream.flush timed out
IOStream.flush timed out
IOStream.flush timed out
IOStream.flush timed out
IOStream.flush timed out
IOStream.flush timed out
IOStream.flush timed out
IOStream.flush timed out
IOStream.flush timed out
IOStream.flush timed out
IOStream.flush timed out
IOStream.flush timed out
IOStream.flush timed out
IOStream.flush timed out
IOStream.flush timed out
IOStream.flush timed out
IOStream.flush timed out
IOStream.flush timed out


In [51]:
adata.write_h5ad('./processed/ann_cell/ann_cell_cellhint.h5ad', compression='gzip')

... storing 'low_hierarchy' as categorical
... storing 'high_hierarchy' as categorical


In [ ]:
sc.pl.umap(adata, color = ['dataset'])

In [ ]:
sc.pl.umap(adata, color = 'high_hierarchy')

## harmony

In [2]:
adata = sc.read_h5ad('./processed/ann_cell/ann_cell_rna.h5ad')
adata

AnnData object with n_obs × n_vars = 2113504 × 36601
    obs: 'dataset', 'barcode', 'sample', 'cellType'

In [ ]:
sc.pp.calculate_qc_metrics(adata,inplace=True)
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)
sc.pp.highly_variable_genes(adata, min_mean=0.0125, max_mean=3, min_disp=0.5)
adata.raw = adata
adata = adata[:, adata.var.highly_variable]
sc.pp.scale(adata, max_value=10)
sc.tl.pca(adata, svd_solver='arpack')
sc.pl.pca_variance_ratio(adata, log=True)

/home/qiaolibo/miniconda3/envs/scanpy/lib/python3.11/site-packages/scanpy/preprocessing/_scale.py:316: UserWarning: Received a view of an AnnData. Making a copy.
  view_to_actual(adata)


In [ ]:
adata

In [ ]:
sce.pp.harmony_integrate(adata, 'sample')
adata.obsm['X_pca'] = adata.obsm['X_pca_harmony']
sc.pp.neighbors(adata, n_neighbors=10, n_pcs=40)

adata.write('rna_neighbor.h5ad', compression='gzip')

In [2]:
adata = sc.read_h5ad('rna_neighbor.h5ad')
adata

AnnData object with n_obs × n_vars = 2113504 × 6637
    obs: 'dataset', 'barcode', 'sample', 'cellType', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'pct_counts_in_top_50_genes', 'pct_counts_in_top_100_genes', 'pct_counts_in_top_200_genes', 'pct_counts_in_top_500_genes'
    var: 'n_cells_by_counts', 'mean_counts', 'log1p_mean_counts', 'pct_dropout_by_counts', 'total_counts', 'log1p_total_counts', 'highly_variable', 'means', 'dispersions', 'dispersions_norm', 'mean', 'std'
    uns: 'hvg', 'log1p', 'neighbors', 'pca'
    obsm: 'X_pca', 'X_pca_harmony'
    varm: 'PCs'
    obsp: 'connectivities', 'distances'

In [3]:
sc.tl.umap(adata)
sc.tl.leiden(adata,resolution=1)

adata.write('rna_leiden.h5ad', compression='gzip')

/home/qiaolibo/miniconda3/envs/scanpy/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/qiaolibo/miniconda3/envs/scanpy/lib/python3.11/site-packages/umap/spectral.py:550: UserWarning: Spectral initialisation failed! The eigenvector solver
failed. This is likely due to too small an eigengap. Consider
adding some noise or jitter to your data.

Falling back to random initialisation!
  warn(


KeyboardInterrupt: 

In [31]:
adata = sc.read_h5ad('./processed/ann_cell/rna_leiden.h5ad')

In [33]:
adata_c = sc.read_h5ad('./processed/ann_cell/ann_cell_celltypist.h5ad')
adata_c

AnnData object with n_obs × n_vars = 2113504 × 36601
    obs: 'dataset', 'barcode', 'sample', 'cellType', 'predicted_labels', 'over_clustering', 'majority_voting', 'conf_score'
    uns: 'log1p', 'neighbors', 'over_clustering'
    obsm: 'X_pca'
    obsp: 'connectivities', 'distances'

In [34]:
adata.obs['majority_voting'] = adata_c.obs['majority_voting']

In [ ]:
sc.pl.umap(adata, color='dataset')

In [ ]:
sc.pl.umap(adata, color='majority_voting')

# Filter unannotated cells using all_ann

In [2]:
adata = sc.read_h5ad('./processed/all_cell/rna.h5ad')
adata

AnnData object with n_obs × n_vars = 2556326 × 36601
    obs: 'dataset', 'barcode', 'sample'

In [13]:
ann = pd.read_csv('../4_cell_ann/ann_merge.csv', index_col=0)
ann

,cellType
Chen_Nat_Methods_2022-AAACAGCCAACTGGCT-lane1,Treg
Chen_Nat_Methods_2022-AAACATGCACAATTAC-lane1,Treg
Chen_Nat_Methods_2022-AAACATGCAGCACGTT-lane1,Th17
Chen_Nat_Methods_2022-AAACATGCATATTGAC-lane1,Th1
Chen_Nat_Methods_2022-AAACCAACAATAGCAA-lane1,TCM
...,...
Wimmers_Cell_2023-TTTGTCTAGTTGTCCC-76_CC0829,CD14 Monocyte
Wimmers_Cell_2023-TTTGTGGCAGAGAGCC-76_CC0829,CD8 T Naive
Wimmers_Cell_2023-TTTGTGGCATTGCAGC-76_CC0829,PB
Wimmers_Cell_2023-TTTGTGGCATTGCGAC-76_CC0829,CD14 Monocyte


In [14]:
adata_filtered = adata[~adata.obs_names.isin(ann.index)]
adata_filtered

View of AnnData object with n_obs × n_vars = 410852 × 36601
    obs: 'dataset', 'barcode', 'sample'

In [15]:
adata_filtered.obs['dataset'].value_counts()

dataset
Wei_Immuni_2023                143804
Boukhaled_Nat_Immunol_2022     101020
Mimitou_Nat_Biotechnol_2021     39374
Weng_Nature_2024_6              23154
Swanson_Elife_2021_2            14680
10x_datasets_8                  12016
10x_datasets_6                  12012
10x_datasets_10                 11909
10x_datasets_4                  11898
10x_datasets_1                  10970
10x_datasets_2                  10691
Weng_Nature_2024_7               7878
10x_datasets_7                   3012
10x_datasets_5                   3009
10x_datasets_9                   2714
10x_datasets_3                   2711
Name: count, dtype: int64

In [16]:
adata_filtered.write_h5ad('./processed/no_annotation/all_cell_rna.h5ad', compression='gzip')

## Basic filtering

In [2]:
adata = sc.read_h5ad('./processed/no_annotation/all_cell_rna.h5ad')
adata

AnnData object with n_obs × n_vars = 410852 × 36601
    obs: 'dataset', 'barcode', 'sample'

In [3]:
adata.obs['dataset'].value_counts()

dataset
Wei_Immuni_2023                143804
Boukhaled_Nat_Immunol_2022     101020
Mimitou_Nat_Biotechnol_2021     39374
Weng_Nature_2024_6              23154
Swanson_Elife_2021_2            14680
10x_datasets_8                  12016
10x_datasets_6                  12012
10x_datasets_10                 11909
10x_datasets_4                  11898
10x_datasets_1                  10970
10x_datasets_2                  10691
Weng_Nature_2024_7               7878
10x_datasets_7                   3012
10x_datasets_5                   3009
10x_datasets_9                   2714
10x_datasets_3                   2711
Name: count, dtype: int64

In [4]:
adata.obs['sample'].value_counts()

sample
Wei_Immuni_2023-YW4                     20000
Wei_Immuni_2023-YW9                     18114
Wei_Immuni_2023-YW6                     16042
Wei_Immuni_2023-YW10                    14562
Boukhaled_Nat_Immunol_2022-P2           14493
Wei_Immuni_2023-HD1                     14472
Wei_Immuni_2023-YW1                     14012
Wei_Immuni_2023-YW2                     13440
Boukhaled_Nat_Immunol_2022-P3           13010
Weng_Nature_2024_6-BMMC                 12298
Wei_Immuni_2023-YW5                     12198
10x_datasets_8-1                        12016
10x_datasets_6-1                        12012
Boukhaled_Nat_Immunol_2022-HD2          11952
10x_datasets_10-1                       11909
10x_datasets_4-1                        11898
Mimitou_Nat_Biotechnol_2021-DIG_CTRL    11868
Mimitou_Nat_Biotechnol_2021-DIG_STIM    11651
Wei_Immuni_2023-YW3                     11476
10x_datasets_1-1                        10970
Weng_Nature_2024_6-HSPC                 10856
Boukhaled_Nat_Immunol_2022-

In [5]:
sc.pp.calculate_qc_metrics(adata,inplace=True)

In [6]:
adata.var["mt"] = adata.var_names.str.startswith("MT-")
sc.pp.calculate_qc_metrics(
    adata, qc_vars=["mt"], percent_top=None, log1p=False, inplace=True
)

In [7]:
atac_meta = pd.read_csv('./processed/no_annotation/atac_meta.csv', index_col=0)
atac_meta

,dataset,barcode,sample,n_genes_by_counts,log1p_n_genes_by_counts,total_counts,log1p_total_counts,pct_counts_in_top_50_genes,pct_counts_in_top_100_genes,pct_counts_in_top_200_genes,pct_counts_in_top_500_genes
10x_datasets_1-AAACAGCCAACAACAA-1,10x_datasets_1,AAACAGCCAACAACAA,10x_datasets_1-1,16759,9.726750,22041.0,10.000705,1.370174,2.504424,4.328297,8.965110
10x_datasets_1-AAACAGCCACCGGCTA-1,10x_datasets_1,AAACAGCCACCGGCTA,10x_datasets_1-1,9068,9.112617,11613.0,9.359966,1.851373,3.401361,5.984672,12.916559
10x_datasets_1-AAACAGCCAGGACACA-1,10x_datasets_1,AAACAGCCAGGACACA,10x_datasets_1-1,7949,8.980927,9939.0,9.204322,2.153134,3.903813,6.922226,14.136231
10x_datasets_1-AAACAGCCATCCTAGA-1,10x_datasets_1,AAACAGCCATCCTAGA,10x_datasets_1-1,11413,9.342596,14018.0,9.548169,1.505208,2.739335,4.879441,10.778998
10x_datasets_1-AAACATGCAAAGGTAC-1,10x_datasets_1,AAACATGCAAAGGTAC,10x_datasets_1-1,14329,9.570111,17353.0,9.761579,1.233216,2.253213,3.982020,8.811157
...,...,...,...,...,...,...,...,...,...,...,...
Weng_Nature_2024_7-TTTGTTGGTAGTTACG-1,Weng_Nature_2024_7,TTTGTTGGTAGTTACG,Weng_Nature_2024_7-1,358,5.883322,367.0,5.908083,16.076294,29.700272,56.948229,100.000000
Weng_Nature_2024_7-TTTGTTGGTCATTACC-1,Weng_Nature_2024_7,TTTGTTGGTCATTACC,Weng_Nature_2024_7-1,6559,8.788746,7400.0,8.909370,2.162162,4.000000,6.702703,14.810811
Weng_Nature_2024_7-TTTGTTGGTGTTAAAC-1,Weng_Nature_2024_7,TTTGTTGGTGTTAAAC,Weng_Nature_2024_7-1,3540,8.172164,3918.0,8.273592,3.751914,6.304237,11.408882,22.409393
Weng_Nature_2024_7-TTTGTTGGTTGCATCT-1,Weng_Nature_2024_7,TTTGTTGGTTGCATCT,Weng_Nature_2024_7-1,15849,9.670925,19158.0,9.860528,1.153565,2.176636,3.742562,8.440338


In [8]:
adata.obs['n_peaks'] = atac_meta['n_genes_by_counts'].tolist()
adata.obs['peak_counts'] = atac_meta['total_counts'].tolist()

In [9]:
tss = pd.read_csv('../4_cell_ann/no_ann_tss_merge.csv', index_col=0)
tss

,nucleosome_signal,TSS.enrichment
10x_datasets_1-AAACAGCCAACAACAA-1,0.853455,5.424387
10x_datasets_1-AAACAGCCACCGGCTA-1,0.631136,5.606893
10x_datasets_1-AAACAGCCAGGACACA-1,0.640569,6.125467
10x_datasets_1-AAACAGCCATCCTAGA-1,0.678705,5.772711
10x_datasets_1-AAACATGCAAAGGTAC-1,0.841932,4.390500
...,...,...
Weng_Nature_2024_7-TTTGTTGGTAGTTACG-1,0.552239,5.328005
Weng_Nature_2024_7-TTTGTTGGTCATTACC-1,0.677709,5.041056
Weng_Nature_2024_7-TTTGTTGGTGTTAAAC-1,0.640547,4.649035
Weng_Nature_2024_7-TTTGTTGGTTGCATCT-1,0.641038,4.009854


In [10]:
adata.obs['nucleosome_signal'] = tss['nucleosome_signal'].tolist()
adata.obs['TSS.enrichment'] = tss['TSS.enrichment'].tolist()

In [11]:
adata

AnnData object with n_obs × n_vars = 410852 × 36601
    obs: 'dataset', 'barcode', 'sample', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'pct_counts_in_top_50_genes', 'pct_counts_in_top_100_genes', 'pct_counts_in_top_200_genes', 'pct_counts_in_top_500_genes', 'total_counts_mt', 'pct_counts_mt', 'n_peaks', 'peak_counts', 'nucleosome_signal', 'TSS.enrichment'
    var: 'n_cells_by_counts', 'mean_counts', 'log1p_mean_counts', 'pct_dropout_by_counts', 'total_counts', 'log1p_total_counts', 'mt'

In [ ]:
sc.pl.violin(
    adata,
    ["total_counts", "pct_counts_mt"],
    size=0,
    multi_panel=True,
)

In [ ]:
sc.pl.violin(
    adata,
    ['peak_counts', "nucleosome_signal", 'TSS.enrichment'],
    size=0,
    multi_panel=True,
)

In [14]:
adata = adata[adata.obs.total_counts < 20000, :]
adata = adata[adata.obs.total_counts > 500, :]
adata

View of AnnData object with n_obs × n_vars = 392283 × 36601
    obs: 'dataset', 'barcode', 'sample', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'pct_counts_in_top_50_genes', 'pct_counts_in_top_100_genes', 'pct_counts_in_top_200_genes', 'pct_counts_in_top_500_genes', 'total_counts_mt', 'pct_counts_mt', 'n_peaks', 'peak_counts', 'nucleosome_signal', 'TSS.enrichment'
    var: 'n_cells_by_counts', 'mean_counts', 'log1p_mean_counts', 'pct_dropout_by_counts', 'total_counts', 'log1p_total_counts', 'mt'

In [15]:
adata = adata[adata.obs.pct_counts_mt < 25, :]
adata

View of AnnData object with n_obs × n_vars = 382841 × 36601
    obs: 'dataset', 'barcode', 'sample', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'pct_counts_in_top_50_genes', 'pct_counts_in_top_100_genes', 'pct_counts_in_top_200_genes', 'pct_counts_in_top_500_genes', 'total_counts_mt', 'pct_counts_mt', 'n_peaks', 'peak_counts', 'nucleosome_signal', 'TSS.enrichment'
    var: 'n_cells_by_counts', 'mean_counts', 'log1p_mean_counts', 'pct_dropout_by_counts', 'total_counts', 'log1p_total_counts', 'mt'

In [23]:
adata = adata[adata.obs.peak_counts > 500, :]
adata = adata[adata.obs.peak_counts < 30000, :]
adata

View of AnnData object with n_obs × n_vars = 349124 × 36601
    obs: 'dataset', 'barcode', 'sample', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'pct_counts_in_top_50_genes', 'pct_counts_in_top_100_genes', 'pct_counts_in_top_200_genes', 'pct_counts_in_top_500_genes', 'total_counts_mt', 'pct_counts_mt', 'n_peaks', 'peak_counts', 'nucleosome_signal', 'TSS.enrichment'
    var: 'n_cells_by_counts', 'mean_counts', 'log1p_mean_counts', 'pct_dropout_by_counts', 'total_counts', 'log1p_total_counts', 'mt'

In [24]:
adata = adata[adata.obs['TSS.enrichment'] > 1, :]
adata = adata[adata.obs.nucleosome_signal < 2, :]
adata

View of AnnData object with n_obs × n_vars = 349124 × 36601
    obs: 'dataset', 'barcode', 'sample', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'pct_counts_in_top_50_genes', 'pct_counts_in_top_100_genes', 'pct_counts_in_top_200_genes', 'pct_counts_in_top_500_genes', 'total_counts_mt', 'pct_counts_mt', 'n_peaks', 'peak_counts', 'nucleosome_signal', 'TSS.enrichment'
    var: 'n_cells_by_counts', 'mean_counts', 'log1p_mean_counts', 'pct_dropout_by_counts', 'total_counts', 'log1p_total_counts', 'mt'

In [ ]:
sc.pl.violin(
    adata,
    ["total_counts", "pct_counts_mt"],
    size=0,
    multi_panel=True,
)

In [ ]:
sc.pl.violin(
    adata,
    ['peak_counts', "nucleosome_signal", 'TSS.enrichment'],
    size=0,
    multi_panel=True,
)

In [27]:
adata.obs['dataset'].value_counts()

dataset
Wei_Immuni_2023                116549
Boukhaled_Nat_Immunol_2022      86361
Mimitou_Nat_Biotechnol_2021     32695
Weng_Nature_2024_6              21062
Swanson_Elife_2021_2            11796
10x_datasets_4                  11187
10x_datasets_10                 11179
10x_datasets_6                  11112
10x_datasets_8                  11084
10x_datasets_1                   9665
10x_datasets_2                   9341
Weng_Nature_2024_7               6356
10x_datasets_5                   2791
10x_datasets_7                   2777
10x_datasets_3                   2587
10x_datasets_9                   2582
Name: count, dtype: int64

In [28]:
adata.obs['sample'].value_counts()

sample
Wei_Immuni_2023-YW4                     17123
Wei_Immuni_2023-YW9                     13649
Wei_Immuni_2023-YW6                     13282
Wei_Immuni_2023-HD1                     12327
Boukhaled_Nat_Immunol_2022-P2           11690
Wei_Immuni_2023-YW1                     11583
10x_datasets_4-1                        11187
10x_datasets_10-1                       11179
10x_datasets_6-1                        11112
10x_datasets_8-1                        11084
Boukhaled_Nat_Immunol_2022-P3           11044
Weng_Nature_2024_6-BMMC                 11027
Wei_Immuni_2023-YW2                     10955
Wei_Immuni_2023-YW10                    10825
Wei_Immuni_2023-YW5                     10263
Weng_Nature_2024_6-HSPC                 10035
Boukhaled_Nat_Immunol_2022-HD2           9907
Mimitou_Nat_Biotechnol_2021-DIG_CTRL     9881
10x_datasets_1-1                         9665
Boukhaled_Nat_Immunol_2022-P4            9561
Wei_Immuni_2023-YW3                      9480
10x_datasets_2-1           

In [29]:
adata.write_h5ad('./processed/no_annotation/filter_cell_rna.h5ad', compression='gzip')

# Celltypist_cellhint

In [36]:
adata = sc.read_h5ad('./processed/no_annotation/filter_cell_rna_celltypist_cellhint.h5ad')
adata

AnnData object with n_obs × n_vars = 349124 × 36601
    obs: 'dataset', 'barcode', 'sample', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'pct_counts_in_top_50_genes', 'pct_counts_in_top_100_genes', 'pct_counts_in_top_200_genes', 'pct_counts_in_top_500_genes', 'total_counts_mt', 'pct_counts_mt', 'n_peaks', 'peak_counts', 'nucleosome_signal', 'TSS.enrichment', 'predicted_labels', 'over_clustering', 'majority_voting', 'conf_score'
    var: 'n_cells_by_counts', 'mean_counts', 'log1p_mean_counts', 'pct_dropout_by_counts', 'total_counts', 'log1p_total_counts', 'mt'
    uns: 'log1p', 'neighbors', 'over_clustering', 'umap'
    obsm: 'X_pca', 'X_umap'
    obsp: 'connectivities', 'distances'

In [ ]:
sc.pl.umap(adata, color='dataset')

In [ ]:
sc.pl.umap(adata, color='sample')

In [ ]:
sc.pl.umap(adata, color='majority_voting')

In [ ]:
sc.pl.umap(adata, color='conf_score')

# ATAC modality

In [2]:
folder_path = '../3_rna_atac_cell/data_atac/'
mtx_files = [os.path.join(folder_path, f) for f in os.listdir(folder_path) if f.endswith('.mtx')]

In [3]:
adata_list = [sc.read_mtx(mtx_file) for mtx_file in mtx_files]

In [4]:
folder_path = '../3_rna_atac_cell/data_atac/'
csv_files = [os.path.join(folder_path, f) for f in os.listdir(folder_path) if f.startswith('cell')]

In [5]:
cell_list = [pd.read_csv(csv_file, index_col=0) for csv_file in csv_files]

In [6]:
for i in range(len(adata_list)):
    adata_list[i].var_names = cell_list[i]['cell_id'].tolist()
    adata_list[i] = adata_list[i].T

In [7]:
combined = adata_list[0].concatenate(adata_list[1], index_unique=None)
combined = combined.concatenate(adata_list[2], index_unique=None)
combined = combined.concatenate(adata_list[3], index_unique=None)

/tmp/ipykernel_814923/3815152369.py:1: FutureWarning: Use anndata.concat instead of AnnData.concatenate, AnnData.concatenate is deprecated and will be removed in the future. See the tutorial for concat at: https://anndata.readthedocs.io/en/latest/concatenation.html
  combined = adata_list[0].concatenate(adata_list[1], index_unique=None)
/tmp/ipykernel_814923/3815152369.py:2: FutureWarning: Use anndata.concat instead of AnnData.concatenate, AnnData.concatenate is deprecated and will be removed in the future. See the tutorial for concat at: https://anndata.readthedocs.io/en/latest/concatenation.html
  combined = combined.concatenate(adata_list[2], index_unique=None)
/tmp/ipykernel_814923/3815152369.py:3: FutureWarning: Use anndata.concat instead of AnnData.concatenate, AnnData.concatenate is deprecated and will be removed in the future. See the tutorial for concat at: https://anndata.readthedocs.io/en/latest/concatenation.html
  combined = combined.concatenate(adata_list[3], index_unique

In [ ]:
combined = combined.concatenate(adata_list[4], index_unique=None)
combined = combined.concatenate(adata_list[5], index_unique=None)

/tmp/ipykernel_814923/3695352608.py:1: FutureWarning: Use anndata.concat instead of AnnData.concatenate, AnnData.concatenate is deprecated and will be removed in the future. See the tutorial for concat at: https://anndata.readthedocs.io/en/latest/concatenation.html
  combined = combined.concatenate(adata_list[4], index_unique=None)
/tmp/ipykernel_814923/3695352608.py:2: FutureWarning: Use anndata.concat instead of AnnData.concatenate, AnnData.concatenate is deprecated and will be removed in the future. See the tutorial for concat at: https://anndata.readthedocs.io/en/latest/concatenation.html
  combined = combined.concatenate(adata_list[5], index_unique=None)


In [14]:
adata_list

[AnnData object with n_obs × n_vars = 405023 × 527459,
 AnnData object with n_obs × n_vars = 1031726 × 527459,
 AnnData object with n_obs × n_vars = 185682 × 527459,
 AnnData object with n_obs × n_vars = 480289 × 527459,
 AnnData object with n_obs × n_vars = 136700 × 527459,
 AnnData object with n_obs × n_vars = 316906 × 527459]

In [15]:
405023+1031726+185682+480289+136700+316906

2556326

In [12]:
combined

AnnData object with n_obs × n_vars = 2556326 × 527459
    obs: 'batch'

In [24]:
peak = pd.read_csv('../3_rna_atac_cell/data_atac/peak.csv', index_col=0)
peak

,peak
1,chr1-792623-793123
2,chr1-794173-794673
3,chr1-804657-805157
4,chr1-807626-808126
5,chr1-811266-811766
...,...
527455,chrX-155966772-155967272
527456,chrX-155968031-155968531
527457,chrX-155968730-155969230
527458,chrX-155971566-155972066


In [25]:
combined.var_names = peak['peak'].tolist()

In [17]:
del combined.obs['batch']
combined

AnnData object with n_obs × n_vars = 2556326 × 527459

In [18]:
df = pd.DataFrame([x.split('-') for x in combined.obs_names], columns=['part1', 'part2', 'part3'])

In [19]:
combined.obs['dataset'] = df['part1'].tolist()
combined.obs['barcode'] = df['part2'].tolist()

In [20]:
df['part4'] = df['part1'] + '-' + df['part3']

In [21]:
combined.obs['sample'] = df['part4'].tolist()

In [26]:
combined.var

""
chr1-792623-793123
chr1-794173-794673
chr1-804657-805157
chr1-807626-808126
chr1-811266-811766
...
chrX-155966772-155967272
chrX-155968031-155968531
chrX-155968730-155969230
chrX-155971566-155972066


In [ ]:
combined.write('atac.h5ad', compression='gzip')

## annotated cells

In [3]:
ann = pd.read_csv('./processed/original_ann/ann_filter.csv', index_col=0)
ann

,cellType
Chen_Nat_Methods_2022-AAACAGCCAACTGGCT-lane1,Treg
Chen_Nat_Methods_2022-AAACATGCACAATTAC-lane1,Treg
Chen_Nat_Methods_2022-AAACATGCAGCACGTT-lane1,Th17
Chen_Nat_Methods_2022-AAACATGCATATTGAC-lane1,Th1
Chen_Nat_Methods_2022-AAACCAACAATAGCAA-lane1,TCM
...,...
Wimmers_Cell_2023-TTTGTCTAGTTGTCCC-76_CC0829,CD14 Monocyte
Wimmers_Cell_2023-TTTGTGGCAGAGAGCC-76_CC0829,CD8 T Naive
Wimmers_Cell_2023-TTTGTGGCATTGCAGC-76_CC0829,PB
Wimmers_Cell_2023-TTTGTGGCATTGCGAC-76_CC0829,CD14 Monocyte


In [4]:
adata_filtered = adata[adata.obs_names.isin(ann.index)]
adata_filtered

View of AnnData object with n_obs × n_vars = 2113504 × 527459
    obs: 'dataset', 'barcode', 'sample'

In [5]:
ann_sorted = ann.reindex(adata_filtered.obs_names.tolist())
ann_sorted

,cellType
Chen_Nat_Methods_2022-AAACAGCCAACTGGCT-lane1,Treg
Chen_Nat_Methods_2022-AAACATGCACAATTAC-lane1,Treg
Chen_Nat_Methods_2022-AAACATGCAGCACGTT-lane1,Th17
Chen_Nat_Methods_2022-AAACATGCATATTGAC-lane1,Th1
Chen_Nat_Methods_2022-AAACCAACAATAGCAA-lane1,TCM
...,...
Wimmers_Cell_2023-TTTGTCTAGTTGTCCC-76_CC0829,CD14 Monocyte
Wimmers_Cell_2023-TTTGTGGCAGAGAGCC-76_CC0829,CD8 T Naive
Wimmers_Cell_2023-TTTGTGGCATTGCAGC-76_CC0829,PB
Wimmers_Cell_2023-TTTGTGGCATTGCGAC-76_CC0829,CD14 Monocyte


In [6]:
adata_filtered.obs['cellType'] = ann_sorted['cellType'].tolist()

/tmp/ipykernel_1551140/799291639.py:1: ImplicitModificationWarning: Trying to modify attribute `.obs` of view, initializing view as actual.
  adata_filtered.obs['cellType'] = ann_sorted['cellType'].tolist()


In [7]:
adata_filtered.write_h5ad('./processed/ann_cell/ann_cell_atac.h5ad', compression='gzip')

## unannotated cells

In [9]:
ann = pd.read_csv('../4_cell_ann/ann_merge.csv', index_col=0)
ann

,cellType
Chen_Nat_Methods_2022-AAACAGCCAACTGGCT-lane1,Treg
Chen_Nat_Methods_2022-AAACATGCACAATTAC-lane1,Treg
Chen_Nat_Methods_2022-AAACATGCAGCACGTT-lane1,Th17
Chen_Nat_Methods_2022-AAACATGCATATTGAC-lane1,Th1
Chen_Nat_Methods_2022-AAACCAACAATAGCAA-lane1,TCM
...,...
Wimmers_Cell_2023-TTTGTCTAGTTGTCCC-76_CC0829,CD14 Monocyte
Wimmers_Cell_2023-TTTGTGGCAGAGAGCC-76_CC0829,CD8 T Naive
Wimmers_Cell_2023-TTTGTGGCATTGCAGC-76_CC0829,PB
Wimmers_Cell_2023-TTTGTGGCATTGCGAC-76_CC0829,CD14 Monocyte


In [10]:
adata_filtered = adata[~adata.obs_names.isin(ann.index)]
adata_filtered

View of AnnData object with n_obs × n_vars = 410852 × 527459
    obs: 'dataset', 'barcode', 'sample'

In [11]:
adata_filtered.write_h5ad('./processed/no_annotation/all_cell_atac.h5ad', compression='gzip')

## basic filtering

In [13]:
adata = sc.read_h5ad('./processed/no_annotation/all_cell_atac.h5ad')
adata

AnnData object with n_obs × n_vars = 410852 × 527459
    obs: 'dataset', 'barcode', 'sample'

In [15]:
sc.pp.calculate_qc_metrics(adata,inplace=True)

In [18]:
adata.obs.to_csv('processed/no_annotation/atac_meta.csv')

In [3]:
adata = sc.read_h5ad('./processed/no_annotation/all_cell_atac.h5ad')
adata

AnnData object with n_obs × n_vars = 410852 × 527459
    obs: 'dataset', 'barcode', 'sample'

In [4]:
adata_a = sc.read_h5ad('./processed/no_annotation/filter_cell_rna.h5ad')

In [5]:
adata_a

AnnData object with n_obs × n_vars = 349124 × 36601
    obs: 'dataset', 'barcode', 'sample', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'pct_counts_in_top_50_genes', 'pct_counts_in_top_100_genes', 'pct_counts_in_top_200_genes', 'pct_counts_in_top_500_genes', 'total_counts_mt', 'pct_counts_mt', 'n_peaks', 'peak_counts', 'nucleosome_signal', 'TSS.enrichment'
    var: 'n_cells_by_counts', 'mean_counts', 'log1p_mean_counts', 'pct_dropout_by_counts', 'total_counts', 'log1p_total_counts', 'mt'

In [12]:
adata_filter = adata[adata_a.obs_names.tolist(),:]

In [15]:
adata_filter.write_h5ad('./processed/no_annotation/filter_cell_atac.h5ad', compression='gzip')

# Celltypist and Cellhint for all cell

In [1]:
import celltypist
import cellhint
import os
import scanpy as sc

/home/qiaolibo/miniconda3/envs/scanpy/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
os.chdir('/pool1/qiaolibo/R_Project/peak_gene/z_others/5_matrix_merge/')

## compute

In [2]:
adata = sc.read_h5ad('./processed/ann_cell/ann_cell_rna.h5ad')
adata

AnnData object with n_obs × n_vars = 2113504 × 36601
    obs: 'dataset', 'barcode', 'sample', 'cellType'

In [3]:
sc.pp.normalize_total(adata, target_sum = 1e4)
sc.pp.log1p(adata)

In [ ]:
adata = celltypist.annotate(adata, model = 'Immune_All_Low.pkl', majority_voting = True).to_adata()

🔬 Input data has 2113504 cells and 36601 genes
🔗 Matching reference genes in the model
🧬 6146 features used for prediction
⚖️ Scaling input data
🖋️ Predicting labels
✅ Prediction done!
👀 Can not detect a neighborhood graph, will construct one before the over-clustering
⛓️ Over-clustering input data with resolution set to 30


In [ ]:
adata

In [ ]:
adata.obs[['predicted_labels', 'majority_voting', 'conf_score']]

In [ ]:
adata.write_h5ad('./processed/ann_cell/ann_cell_celltypist.h5ad', compression='gzip')

In [ ]:
cellhint.integrate(adata, 'sample', 'majority_voting')

In [ ]:
sc.tl.umap(adata)

In [ ]:
adata.write_h5ad('./processed/ann_cell/ann_cell_celltypist_cellhint.h5ad', compression='gzip')

In [ ]:
sc.pl.umap(adata, color = ['dataset', 'sample'], wspace = 0.5)

In [ ]:
sc.pl.umap(adata, color = 'majority_voting')

## check

In [2]:
adata = sc.read_h5ad('./processed/ann_cell/ann_cell_celltypist_cellhint.h5ad')
adata

AnnData object with n_obs × n_vars = 2113504 × 36601
    obs: 'dataset', 'barcode', 'sample', 'cellType', 'predicted_labels', 'over_clustering', 'majority_voting', 'conf_score'
    uns: 'log1p', 'neighbors', 'over_clustering', 'umap'
    obsm: 'X_pca', 'X_umap'
    obsp: 'connectivities', 'distances'

In [7]:
adata.obs

,dataset,barcode,sample,cellType,predicted_labels,over_clustering,majority_voting,conf_score
Chen_Nat_Methods_2022-AAACAGCCAACTGGCT-lane1,Chen_Nat_Methods_2022,AAACAGCCAACTGGCT,Chen_Nat_Methods_2022-lane1,Treg,Regulatory T cells,59,Tem/Effector helper T cells,0.511893
Chen_Nat_Methods_2022-AAACATGCACAATTAC-lane1,Chen_Nat_Methods_2022,AAACATGCACAATTAC,Chen_Nat_Methods_2022-lane1,Treg,Tem/Effector helper T cells,117,Tcm/Naive helper T cells,0.011004
Chen_Nat_Methods_2022-AAACATGCAGCACGTT-lane1,Chen_Nat_Methods_2022,AAACATGCAGCACGTT,Chen_Nat_Methods_2022-lane1,Th17,CD16- NK cells,62,Tcm/Naive helper T cells,0.022152
Chen_Nat_Methods_2022-AAACATGCATATTGAC-lane1,Chen_Nat_Methods_2022,AAACATGCATATTGAC,Chen_Nat_Methods_2022-lane1,Th1,Tem/Trm cytotoxic T cells,62,Tcm/Naive helper T cells,0.953910
Chen_Nat_Methods_2022-AAACCAACAATAGCAA-lane1,Chen_Nat_Methods_2022,AAACCAACAATAGCAA,Chen_Nat_Methods_2022-lane1,TCM,Tcm/Naive helper T cells,296,Tcm/Naive helper T cells,0.222813
...,...,...,...,...,...,...,...,...
Hu_BioRxiv_2023-T124.R1.192.R2.191.R3.121.P1.04-donor7,Hu_BioRxiv_2023,T124.R1.192.R2.191.R3.121.P1.04,Hu_BioRxiv_2023-donor7,CD4,Tcm/Naive helper T cells,187,Plasma cells,0.856617
Hu_BioRxiv_2023-T124.R1.192.R2.191.R3.135.P1.04-donor7,Hu_BioRxiv_2023,T124.R1.192.R2.191.R3.135.P1.04,Hu_BioRxiv_2023-donor7,NK,Late erythroid,142,Plasma cells,0.906738
Hu_BioRxiv_2023-T124.R1.192.R2.192.R3.001.P1.04-donor7,Hu_BioRxiv_2023,T124.R1.192.R2.192.R3.001.P1.04,Hu_BioRxiv_2023-donor7,CD4,Proliferative germinal center B cells,4,Tcm/Naive helper T cells,0.983643
Hu_BioRxiv_2023-T124.R1.192.R2.192.R3.033.P1.04-donor7,Hu_BioRxiv_2023,T124.R1.192.R2.192.R3.033.P1.04,Hu_BioRxiv_2023-donor7,CD8,Plasma cells,91,Late erythroid,0.974617


In [ ]:
sc.pl.umap(adata, color='dataset')

In [ ]:
sc.pl.umap(adata, color='majority_voting')

In [ ]:
sc.pl.umap(adata, color='predicted_labels')

In [ ]:
sc.pl.umap(adata, color='conf_score')